# **Library Stuff**

In [2]:
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 1.2 MB/s eta 0:00:00


In [23]:
!pip install torchao -U

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 10.3 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [3]:
from datasets import load_dataset, DatasetDict,Dataset
from transformers import (
    AutoTokenizer,
    AutoConfig,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
    pipeline)
from peft import PeftModel, PeftConfig, get_peft_model, LoraConfig
import evaluate
import torch
import numpy as np

# **Dataset**

In [7]:
dataset = load_dataset('shawhin/imdb-truncated')

README.md:   0%|          | 0.00/592 [00:00<?, ?B/s]

data/train-00000-of-00001-5a744bf76a1d84(…):   0%|          | 0.00/836k [00:00<?, ?B/s]

data/validation-00000-of-00001-a3a52fabb(…):   0%|          | 0.00/853k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [9]:
train_df = dataset['train'].to_pandas()
val_df = dataset['validation'].to_pandas()
print('Train DF')
print(train_df.head(3))
print('Test DF')
print(val_df.head(3))

Train DF
   label                                               text
0      1  . . . or type on a computer keyboard, they'd p...
1      1  During 1933 this film had many cuts taken from...
2      0  Let me be clear. I've used IMDb for years. But...
Test DF
   label                                               text
0      1  Disgused as an Asian Horror, "A Tale Of Two Si...
1      1  I am from Texas and my family vacationed a cou...
2      0  Robert Altman's "Quintet" is a dreary, gloomy,...


# **Training Part**

In [10]:
model_checkpoint= 'FacebookAI/roberta-base'

In [5]:
id2label = {0: 'NEGATIVE', 1: 'POSITIVE'}
label2id = {'NEGATIVE': 0, 'POSITIVE': 1}

In [11]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_checkpoint,num_labels=2,
    id2label=id2label,
    label2id=label2id
)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: FacebookAI/roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.weight      | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


**Preprocessing**

In [12]:
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint, add_prefix_space=True)

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [13]:
def tokenize_function(data):
  text = data['text']
  tokenizer.truncation_side='left'
  tokenized_inputs = tokenizer(
      text,
      return_tensors='np',
      truncation=True,
      max_length=512,
  )
  return tokenized_inputs
if tokenizer.pad_token is None:
  tokenizer.add_special_tokens({'pad_token': '[PAD]'})
  model.resize_token_embeddings(len(tokenizer))
tokenized_dataset = dataset.map(tokenize_function, batched=True)
tokenized_dataset

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['label', 'text', 'input_ids', 'attention_mask'],
        num_rows: 1000
    })
    validation: Dataset({
        features: ['label', 'text', 'input_ids', 'attention_mask'],
        num_rows: 1000
    })
})

In [19]:
tokenized_dataset['train'].to_pandas().head(2)

,label,text,input_ids,attention_mask
0,1,". . . or type on a computer keyboard, they'd p...","[0, 479, 479, 479, 50, 1907, 15, 10, 3034, 127...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
1,1,During 1933 this film had many cuts taken from...,"[0, 1590, 26873, 42, 822, 56, 171, 2599, 551, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."


In [29]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# **Evaluation Metrics**

In [20]:
accuracy = evaluate.load('accuracy')
def compute_metrics(p):
  predictions, labels = p
  predictions = np.argmax(predictions, axis=1)
  return {'accuracy': accuracy.compute(predictions=predictions, references=labels)}

# **LoRA Configurations**

In [21]:
peft_config = LoraConfig(
    task_type = "SEQ_CLS",#sequence classification
    r=4,
    lora_alpha = 8,
    lora_dropout=0.1,
    target_modules=['query']
)
#

In [25]:
model_peft = get_peft_model(model,peft_config)
model_peft.print_trainable_parameters()

trainable params: 665,858 || all params: 125,313,028 || trainable%: 0.5314


/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:302: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


# **Hyperparameters**

In [28]:
lr= 1e-3
batch_size =4
num_epochs =10
training_args = TrainingArguments(
    output_dir='./Roberta_Classification_results',
    learning_rate=lr,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    num_train_epochs=num_epochs,
    weight_decay=0.01,
    eval_strategy = 'epoch',
    save_strategy = 'epoch',
    load_best_model_at_end=True,
)

In [33]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics
)
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy
1,No log,0.236746,{'accuracy': 0.939}
2,0.321221,0.390532,{'accuracy': 0.924}
3,0.321221,0.427849,{'accuracy': 0.928}
4,0.123679,0.387766,{'accuracy': 0.938}
5,0.123679,0.497819,{'accuracy': 0.935}
6,0.051166,0.466601,{'accuracy': 0.938}
7,0.051166,0.556310,{'accuracy': 0.937}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

TrainOutput(global_step=2500, training_loss=0.10277801027297974, metrics={'train_runtime': 37366.5881, 'train_samples_per_second': 0.268, 'train_steps_per_second': 0.067, 'total_flos': 2154158913873120.0, 'train_loss': 0.10277801027297974, 'epoch': 10.0})

In [35]:
text_list = ["It was good.", "Not a fan, don't recommed.", "Better than the first one.", "This is not worth watching even once.", "This one is a pass."]
for text in text_list:
  inputs = tokenizer.encode(text, return_tensors='pt')
  logits = model(inputs).logits
  predictions = torch.max(logits,1).indices
  print(text+"----"+ id2label[predictions.tolist()[0]])

It was good.----POSITIVE
Not a fan, don't recommed.----NEGATIVE
Better than the first one.----NEGATIVE
This is not worth watching even once.----NEGATIVE
This one is a pass.----NEGATIVE
